# Dataset 5 — Cleaning
**German adolescents — produces ds5_behavioral_clean.csv, ds5_pre_survey_clean.csv, ds5_post_survey_clean.csv, ds5_participant_linkage.csv**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from zoneinfo import ZoneInfo
from datetime import timezone

RAW5 = Path("../data_raw/5. data set, German adolescents")
BEH5  = RAW5 / "Behavior data set/behavior_data_set_5.csv"
PRE5  = RAW5 / "Survey data sets/Pre-survey/pre_survey_data_set_5.csv"
POST5 = RAW5 / "Survey data sets/Post-survey/post_survey_data_set_5.csv"
OUT   = Path("../outputs/cleaned")
OUT.mkdir(parents=True, exist_ok=True)
print("Paths configured")

## 1. Load Raw Files

In [ ]:
beh  = pd.read_csv(BEH5,  sep=';', low_memory=False)
pre  = pd.read_csv(PRE5,  sep=';', low_memory=False)
post = pd.read_csv(POST5, sep=';', low_memory=False)
print(f"Behavioral: {beh.shape}")
print(f"Pre-survey: {pre.shape}")
print(f"Post-survey: {post.shape}")

## 2. Behavioral — Canonical ID

In [ ]:
PREFIX = "study-one-sec-next-gen-bzkj-2024-"
beh['participant_id_raw'] = beh['userid']
beh['participant_id_canonical'] = beh['userid'].str.removeprefix(PREFIX)
print("Sample canonical IDs:")
print(beh['participant_id_canonical'].head(3).tolist())
print(f"Unique participants: {beh['participant_id_canonical'].nunique()}")

## 3. Behavioral — Timestamp Localization → UTC

In [ ]:
beh['event_timestamp_local'] = pd.to_datetime(beh['timestamp'], errors='coerce')

def localize_to_utc(row):
    ts = row['event_timestamp_local']
    tz_str = row['timezone']
    if pd.isna(ts):
        return pd.NaT
    try:
        tz = ZoneInfo(str(tz_str)) if not pd.isna(tz_str) else ZoneInfo('Europe/Berlin')
        return ts.replace(tzinfo=tz).astimezone(timezone.utc)
    except Exception:
        return pd.NaT

print("Localizing timestamps... (may take ~2 minutes)")
beh['event_timestamp_utc'] = beh.apply(localize_to_utc, axis=1)
beh['event_timestamp_utc'] = pd.to_datetime(beh['event_timestamp_utc'], utc=True)

parse_failures = beh['event_timestamp_utc'].isna().sum()
print(f"Parse failures: {parse_failures}")
print("Sample UTC timestamps:", beh['event_timestamp_utc'].head(3).tolist())

## 4. Behavioral — Event Outcome Mapping

In [ ]:
OUTCOME_MAP = {
    'openedApp': 'opened',
    'dismissedAppOpening': 'dismissed',
    'closedApp': 'closed',
    'dismissedByStructuredIntervention': 'dismissed_other',
}
beh['event_outcome_raw'] = beh['resolution']
beh['event_outcome_simple'] = beh['resolution'].map(OUTCOME_MAP).fillna('missing')
print(beh['event_outcome_simple'].value_counts(dropna=False))

## 5. Behavioral — Intervention Fields

In [ ]:
beh['intervention_duration_sec'] = pd.to_numeric(beh['interventionDuration'], errors='coerce')
beh['reintervention_delay_sec']   = pd.to_numeric(beh['timeIntervalUntilReIntervention'], errors='coerce')
beh['is_intervention_event'] = (beh['intervention_duration_sec'].notna() &
                                 (beh['intervention_duration_sec'] > 0)).astype(int)
beh['is_reintervention'] = beh['reintervention_delay_sec'].notna().astype(int)

print("is_intervention_event:", beh['is_intervention_event'].value_counts().to_dict())
print("is_reintervention:", beh['is_reintervention'].value_counts().to_dict())

## 6. Behavioral — days_since_first_event

In [ ]:
first_ev = beh.sort_values('event_timestamp_utc').groupby('participant_id_canonical')['event_timestamp_utc'].transform('min')
beh['days_since_first_event'] = (beh['event_timestamp_utc'] - first_ev).dt.total_seconds() / 86400
print(f"days_since_first_event: min={beh['days_since_first_event'].min():.1f}, max={beh['days_since_first_event'].max():.1f}")

## 7. Behavioral — Select and Rename Columns

In [ ]:
beh.rename(columns={'app': 'app_name', 'timezone': 'event_timezone'}, inplace=True)

BEH_COLS = [
    'participant_id_raw', 'participant_id_canonical',
    'event_timestamp_utc', 'event_timestamp_local', 'event_timezone',
    'app_name', 'event_outcome_raw', 'event_outcome_simple',
    'intervention_duration_sec', 'reintervention_delay_sec',
    'is_intervention_event', 'is_reintervention',
    'days_since_first_event', 'reason',
    'suggestionBannerShown', 'suggestionDetailViewAppear',
    'suggestionDetailViewDisappear', 'terminationReason',
    'suggestionLearnMoreLinkTapped',
]
beh_clean = beh[BEH_COLS].copy()
print(f"Behavioral clean: {beh_clean.shape}")

## 8. Behavioral — Validation

In [ ]:
assert beh_clean.duplicated().sum() == 0, "Duplicate rows found!"
assert beh_clean['participant_id_canonical'].notna().all(), "Missing canonical IDs!"
assert beh_clean['event_outcome_simple'].isin(['opened','dismissed','closed','dismissed_other','missing']).all()
print("All validation checks passed.")
print(beh_clean['event_outcome_simple'].value_counts())

## 9. Save Behavioral

In [ ]:
beh_clean.to_csv(OUT / "ds5_behavioral_clean.csv", index=False)
print(f"Saved: ds5_behavioral_clean.csv  ({beh_clean.shape})")

## 10. Pre-Survey — Scale Computation

In [ ]:
belong_cols    = [f'belong_{i}'    for i in range(1, 9)]
ostracism_cols = [f'ostracism_{i}' for i in range(1, 6)]
fomo_cols      = [f'fomo_{i}'      for i in range(1, 11)]

for col in belong_cols + ostracism_cols + fomo_cols:
    pre[col] = pd.to_numeric(pre[col], errors='coerce')

pre['belong_mean']    = pre[belong_cols].mean(axis=1)
pre['ostracism_mean'] = pre[ostracism_cols].mean(axis=1)
pre['fomo_mean']      = pre[fomo_cols].mean(axis=1)

for scale, col in [('belong', 'belong_mean'), ('ostracism', 'ostracism_mean'), ('fomo', 'fomo_mean')]:
    print(f"{scale}: mean={pre[col].mean():.2f}, sd={pre[col].std():.2f}, NaN={pre[col].isna().sum()}")

## 11. Pre-Survey — Canonical IDs and Numeric Fields

In [ ]:
pre['participant_id_raw'] = pre['uuid']
pre['participant_id_canonical'] = pre['uuid']

for col in ['digital_wellbeing','time','prob_peers','time_peers','prob_self',
            'social_media_wellbeing','social_media_updated','age']:
    pre[col] = pd.to_numeric(pre[col], errors='coerce')

# ds5pipe-04: plausibility gate. `time` is self-reported HOURS PER DAY on social media.
# 64 of 773 pre-survey responses exceed 24 (distinct values up to 275) -- physically
# impossible, almost certainly minutes entered as hours or typos. Set to NaN rather than
# letting them destroy hours_change downstream: as shipped they reversed its sign in two
# of three arms (-3.568 / -2.464 / -2.603 vs +0.362 / -0.275 / +0.061 once gated).
_n_impossible_pre = int((pre['time'] > 24).sum())
pre.loc[pre['time'] > 24, 'time'] = np.nan
print(f"pre 'time' set to NaN for {_n_impossible_pre} responses above 24 h/day")

print(f"Pre-survey: {pre.shape}")
print(f"Unique UUIDs: {pre['uuid'].nunique()}")
print(f"Condition distribution: {pre['cond'].value_counts().to_dict()}")

## 12. Save Pre-Survey

In [ ]:
pre.to_csv(OUT / "ds5_pre_survey_clean.csv", index=False)
print(f"Saved: ds5_pre_survey_clean.csv  ({pre.shape})")

## 13. Post-Survey — Dedup and Scale Computation

In [ ]:
belong_cols    = [f'belong_{i}'    for i in range(1, 9)]
ostracism_cols = [f'ostracism_{i}' for i in range(1, 6)]
fomo_cols      = [f'fomo_{i}'      for i in range(1, 11)]

for col in belong_cols + ostracism_cols + fomo_cols:
    post[col] = pd.to_numeric(post[col], errors='coerce')

post['belong_mean']    = post[belong_cols].mean(axis=1)
post['ostracism_mean'] = post[ostracism_cols].mean(axis=1)
post['fomo_mean']      = post[fomo_cols].mean(axis=1)

# Dedup — 3 duplicate UUIDs
print(f"Before dedup: {post.shape}")
dupes = post[post.duplicated(subset=['uuid'], keep=False)]['uuid'].value_counts()
print("Duplicate UUIDs:", dupes.to_dict())
# ds5pipe-08: completeness-ranked dedup. The 3 duplicated UUIDs are NOT near-identical
# submissions -- non-null field counts (of 33 raw fields) are 9 vs 33, 5 vs 33 and 33 vs 1,
# so keep='first' discarded the COMPLETE submission in 2 of the 3 cases and left those
# participants' post-survey scales NaN. Keep the most complete row instead; mergesort is
# stable, so ties still fall back to first occurrence and the third case is unchanged.
post_clean = (
    post.assign(_n_filled=post.notna().sum(axis=1))
        .sort_values('_n_filled', ascending=False, kind='mergesort')
        .drop_duplicates(subset=['uuid'], keep='first')
        .drop(columns='_n_filled')
        .sort_index()
        .copy()
)

# ds5pipe-04: same plausibility gate as the pre-survey (see section 11). 15 of the
# retained post-survey responses report more than 24 h/day of social media use (raw max
# 45). Applied AFTER the dedup above because both findings edit `post_clean`, and one of
# the two recovered rows supplies a `time` value that did not exist under keep='first'.
post_clean['time'] = pd.to_numeric(post_clean['time'], errors='coerce')
_n_impossible_post = int((post_clean['time'] > 24).sum())
post_clean.loc[post_clean['time'] > 24, 'time'] = np.nan
print(f"post 'time' set to NaN for {_n_impossible_post} responses above 24 h/day")

post_clean['participant_id_raw'] = post_clean['uuid']
post_clean['participant_id_canonical'] = post_clean['uuid']
print(f"After dedup: {post_clean.shape}")

## 14. Save Post-Survey

In [ ]:
post_clean.to_csv(OUT / "ds5_post_survey_clean.csv", index=False)
print(f"Saved: ds5_post_survey_clean.csv  ({post_clean.shape})")

## 15. Participant Linkage

In [ ]:
beh_ids  = set(beh_clean['participant_id_canonical'].unique())
pre_ids  = set(pre['participant_id_canonical'].unique())
post_ids = set(post_clean['participant_id_canonical'].unique())
all_ids  = beh_ids | pre_ids | post_ids

cond_map = pre.set_index('participant_id_canonical')['cond'].to_dict()

ev_counts = beh_clean.groupby('participant_id_canonical').agg(
    n_events=('event_outcome_simple', 'count'),
    n_opened=('event_outcome_simple', lambda x: (x=='opened').sum()),
    n_dismissed=('event_outcome_simple', lambda x: (x=='dismissed').sum()),
    n_closed=('event_outcome_simple', lambda x: (x=='closed').sum()),
    days_observed=('days_since_first_event', 'max'),
).reset_index()

linkage = pd.DataFrame({'participant_id_canonical': sorted(all_ids)})
linkage['in_behavioral'] = linkage['participant_id_canonical'].isin(beh_ids)
linkage['in_pre_survey'] = linkage['participant_id_canonical'].isin(pre_ids)
linkage['in_post_survey'] = linkage['participant_id_canonical'].isin(post_ids)
linkage['study_condition'] = linkage['participant_id_canonical'].map(cond_map)
linkage = linkage.merge(ev_counts, on='participant_id_canonical', how='left')

linkage.to_csv(OUT / "ds5_participant_linkage.csv", index=False)
print(f"Saved: ds5_participant_linkage.csv  ({linkage.shape})")
print(linkage[['in_behavioral','in_pre_survey','in_post_survey']].value_counts())

## 16. Final Summary

In [ ]:
print("""
=== DS5 CLEANING COMPLETE ===

Outputs:
  ds5_behavioral_clean.csv  : 288,688 rows × 19 cols
  ds5_pre_survey_clean.csv  : 792 rows × 42 cols
  ds5_post_survey_clean.csv : 311 rows × 38 cols (3 post-survey dupes removed)
  ds5_participant_linkage.csv : 793 total participants

Key transformations:
  - userid prefix stripped → participant_id_canonical
  - timestamp localized via timezone col → UTC
  - resolution → event_outcome_simple (5 classes)
  - Scale means computed: belong_mean, ostracism_mean, fomo_mean
  - days_since_first_event derived

Caveats:
  - cond not in behavioral (join from pre-survey)
  - reason is German free text (not categorized)
  - suggestion columns are 99%+ NaN
  - 220 pre-survey participants have no behavioral data
""")